<a href="https://colab.research.google.com/github/Ronak004/Large-Scale-Graph-Optimization-and-Reinforcement-Learning/blob/main/SC3000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**1.1 Set up**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/SC3000_2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SC3000_2


In [ ]:
# Import relevant libraries
import json

In [ ]:
def load_data():
    #Load required files
    with open('G.json', 'r') as f:
        graph = json.load(f)
    with open('Dist.json', 'r') as f:
        dist_map = json.load(f)
    with open('Cost.json', 'r') as f:
        cost_map = json.load(f)
    # Coord is not strictly needed for Dijkstra but may be used for Task 3 heuristics
    return graph, dist_map, cost_map

#**Question 1**

##**Task 1**

Since this is a standard shortest-path problem on a weighted graph, the most efficient and common algorithm to use is **Dijkstra's Algorithm**.

##Initialisation

In [ ]:
# Import relevant libraries
import heapq

##Dijkstra Algorithm

In [ ]:
def dijkstra(start_node, end_node, graph, dist_map, cost_map):
    # priority_queue stores (cumulative_distance, current_node)
    pq = [(0, start_node)]

    # distances[node] stores the minimum distance from start to node
    distances = {node: float('inf') for node in graph}
    distances[start_node] = 0

    # parent[node] stores the predecessor to reconstruct the path
    parent = {start_node: None}

    while pq:
        current_dist, u = heapq.heappop(pq)

        # If we reached the target, we can stop early
        if u == end_node:
            break

        # Standard Dijkstra: if we found a longer path, skip
        if current_dist > distances[u]:
            continue

        # Explore neighbors
        for v in graph.get(u, []):
            weight = dist_map.get(f"{u},{v}", float('inf'))
            new_dist = current_dist + weight

            if new_dist < distances[v]:
                distances[v] = new_dist
                parent[v] = u
                heapq.heappush(pq, (new_dist, v))

    # Reconstruct the path from end_node to start_node
    path = []
    curr = end_node
    total_energy = 0

    if distances[end_node] == float('inf'):
        return None, float('inf'), 0

    while curr is not None:
        path.append(curr)
        prev = parent[curr]
        if prev is not None:
            # Calculate energy cost along the found shortest path
            total_energy += cost_map.get(f"{prev},{curr}", 0)
        curr = prev

    path.reverse()
    return path, distances[end_node], total_energy

##Main Function

In [ ]:
def main():
    # Define start and end nodes as per Task 1 requirements
    start_node = '1'
    end_node = '50'

    print("Loading data files...")
    graph, dist_map, cost_map = load_data()

    print(f"Solving Task 1: Shortest path from {start_node} to {end_node}...")
    path, total_dist, total_energy = dijkstra(start_node, end_node, graph, dist_map, cost_map)

    if path:
        # Format output as required: Shortest path: 1->...->50
        path_str = "->".join(path)
        print(f"Shortest path: {path_str}")
        print(f"Shortest distance: {total_dist}")
        print(f"Total energy cost: {total_energy}")
    else:
        print("No path found.")

In [ ]:
if __name__ == "__main__":
    main()

Loading data files...
Solving Task 1: Shortest path from 1 to 50...
Shortest path: 1->1363->1358->1357->1356->1276->1273->1277->1269->1267->1268->1284->1283->1282->1255->1253->1260->1259->1249->1246->963->964->962->1002->952->1000->998->994->995->996->987->988->979->980->969->977->989->990->991->2369->2366->2340->2338->2339->2333->2334->2329->2029->2027->2019->2022->2000->1996->1997->1993->1992->1989->1984->2001->1900->1875->1874->1965->1963->1964->1923->1944->1945->1938->1937->1939->1935->1931->1934->1673->1675->1674->1837->1671->1828->1825->1817->1815->1634->1814->1813->1632->1631->1742->1741->1740->1739->1591->1689->1585->1584->1688->1579->1679->1677->104->5680->5418->5431->5425->5424->5422->5413->5412->5411->66->5392->5391->5388->5291->5278->5289->5290->5283->5284->5280->50
Shortest distance: 148648.63722140007
Total energy cost: 294853


##**Task 2**: You will need to implement an uninformed search algorithm (e.g., the DFS, BFS, UCS) to solve the NYC instance.

In [ ]:
# Import relevant libraries
import heapq

In [ ]:
# Implementation of a priority queue using a minimising heap with the heapq library
class PriorityQueue:
    def  __init__(self):
        self.heap = []
        self.count = 0

    def push(self, priority, item):
        entry = (priority, item)
        heapq.heappush(self.heap, entry)
        self.count += 1

    def pop(self):
        (priority, item) = heapq.heappop(self.heap)
        return priority, item

    def isEmpty(self):
        return len(self.heap) == 0

In [ ]:
# Implementation of Uniform Cost Search, taking into consideration the energy budget
def uniformCostSearch(graph, start, goal, energy_budget):
    # Initialize the priority queue with the starting node
    priority_queue = PriorityQueue()
    start = (start, 0) # Initialise start node as a tuple: (node, accumulated_energy)
    priority_queue.push(0, start)
    visited = {} # Keep track of the resulting path with format of (node, accumulated_energy) : accumulated_distance

    while not priority_queue.isEmpty():
        # Pop the node with the lowest accumulated distance from the priority queue
        accumulated_distance, current_node = priority_queue.pop()
        node, accumulated_energy = current_node[0], current_node[1]

        # If the current state is the goal, return the visited nodes
        if node == goal:
          visited[current_node] = accumulated_distance
          return visited

        # If the current state hasn't been visited or found at a lower cost
        if current_node not in visited or visited[current_node] > accumulated_distance:
            visited[current_node] = accumulated_distance

            # Explore the current node's neighbours
            for neighbour in graph[node]:
              distance_to_neighbour = accumulated_distance + Dist[node, neighbour]
              energy_to_neighbour = accumulated_energy + Cost[node, neighbour]
              # Check if the neighbour has already been visited, or if there is a shorter path to the neighbour compared to a previous path
              if neighbour not in visited or visited[neighbour] > distance_to_neighbour:
                # Add the neighbour to the priority queue if its addition to the path would not exceed the energy budget
                if energy_to_neighbour <= energy_budget:
                  priority_queue.push(distance_to_neighbour, (neighbour, energy_to_neighbour))

    return {}  # If no path is found

In [ ]:
def costructPath(visited):
  if visited == {}:
    return "No path found"

##Task 3: A* Search with Energy Budget

Import libraries and load remaining required data

In [ ]:
import heapq
import math

def load_coord():
    # Load node coordinates for A* heuristic
    with open('Coord.json', 'r') as f:
        coord_map = json.load(f)

    return coord_map

Define the A* heuristic

In [ ]:
def euclidean_heuristic(node, goal, coord_map):
    x1, y1 = coord_map[node]
    x2, y2 = coord_map[goal]
    return math.hypot(x1 - x2, y1 - y2)

Build the reverse graph

In [ ]:
def build_reverse_graph(graph):
    reverse_graph = {}

    for u, neighbors in graph.items():
        for v in neighbors:
            if v not in reverse_graph:
                reverse_graph[v] = []
            reverse_graph[v].append(u)

    return reverse_graph

Precompute minimum remaining energy to the goal

In [ ]:
def reverse_dijkstra_min_energy(goal, reverse_graph, cost_map):
    min_energy = {goal: 0}
    pq = [(0, goal)]

    while pq:
        curr_energy, node = heapq.heappop(pq)

        if curr_energy != min_energy[node]:
            continue

        for pred in reverse_graph.get(node, []):
            edge_key = f"{pred},{node}"
            new_energy = curr_energy + cost_map[edge_key]

            if pred not in min_energy or new_energy < min_energy[pred]:
                min_energy[pred] = new_energy
                heapq.heappush(pq, (new_energy, pred))

    return min_energy

Dominance checking for constrained search

In [ ]:
def is_dominated(state_list, new_dist, new_cost):
    for old_cost, old_dist, _ in state_list:
        if old_cost <= new_cost and old_dist <= new_dist:
            return True
    return False


def remove_dominated(state_list, new_dist, new_cost):
    filtered = []

    for old_cost, old_dist, sid in state_list:
        if not (new_cost <= old_cost and new_dist <= old_dist):
            filtered.append((old_cost, old_dist, sid))

    return filtered

Reconstruct the final path

In [ ]:
def reconstruct_path(parent, state_info, goal_sid):
    path = []
    sid = goal_sid

    while sid is not None:
        node, _, _ = state_info[sid]
        path.append(node)
        sid = parent[sid]

    path.reverse()
    return path

Main A* algorithm with energy budget

In [ ]:
def astar_with_energy_budget(graph, dist_map, cost_map, coord_map, start, goal, budget):
    reverse_graph = build_reverse_graph(graph)
    min_energy_to_goal = reverse_dijkstra_min_energy(goal, reverse_graph, cost_map)

    # If even the minimum energy needed from start exceeds budget, no solution exists
    if start not in min_energy_to_goal or min_energy_to_goal[start] > budget:
        return None

    pq = []
    parent = {}
    state_info = {}
    label_sets = {}   # node -> list of (energy, distance, sid)

    sid_counter = 0
    start_sid = sid_counter
    sid_counter += 1

    start_g = 0.0
    start_c = 0.0
    start_h = euclidean_heuristic(start, goal, coord_map)

    heapq.heappush(pq, (start_g + start_h, start_g, start_c, start, start_sid))
    parent[start_sid] = None
    state_info[start_sid] = (start, start_g, start_c)
    label_sets[start] = [(start_c, start_g, start_sid)]

    expanded_states = 0

    while pq:
        f, curr_dist, curr_cost, u, sid = heapq.heappop(pq)

        # Skip stale states that were later dominated
        valid = False
        for saved_cost, saved_dist, saved_sid in label_sets.get(u, []):
            if saved_sid == sid and saved_cost == curr_cost and saved_dist == curr_dist:
                valid = True
                break

        if not valid:
            continue

        expanded_states += 1

        # Goal reached
        if u == goal:
            path = reconstruct_path(parent, state_info, sid)
            return {
                "path": path,
                "distance": curr_dist,
                "energy": curr_cost,
                "expanded_states": expanded_states
            }

        # Expand neighbors
        for v in graph[u]:
            edge_key = f"{u},{v}"

            new_dist = curr_dist + dist_map[edge_key]
            new_cost = curr_cost + cost_map[edge_key]

            # Budget check
            if new_cost > budget:
                continue

            # Feasibility pruning using minimum remaining energy
            if v not in min_energy_to_goal:
                continue
            if new_cost + min_energy_to_goal[v] > budget:
                continue

            v_labels = label_sets.get(v, [])

            # Dominance pruning
            if is_dominated(v_labels, new_dist, new_cost):
                continue

            # Remove old labels dominated by the new one
            v_labels = remove_dominated(v_labels, new_dist, new_cost)

            new_sid = sid_counter
            sid_counter += 1

            v_labels.append((new_cost, new_dist, new_sid))
            label_sets[v] = v_labels

            parent[new_sid] = sid
            state_info[new_sid] = (v, new_dist, new_cost)

            h = euclidean_heuristic(v, goal, coord_map)
            heapq.heappush(pq, (new_dist + h, new_dist, new_cost, v, new_sid))

    return None

Run the algorithm

In [ ]:
graph, dist_map, cost_map = load_data()
coord_map = load_coord()

start_node = "1"
goal_node = "50"
energy_budget = 287932

result = astar_with_energy_budget(
    graph=graph,
    dist_map=dist_map,
    cost_map=cost_map,
    coord_map=coord_map,
    start=start_node,
    goal=goal_node,
    budget=energy_budget
)

if result is None:
    print("No feasible path found within the energy budget.")
else:
    print("Shortest path:", "->".join(result["path"]))
    print("Shortest distance:", result["distance"])
    print("Total energy cost:", result["energy"])

Shortest path: 1->1363->1358->1357->1356->1276->1273->1277->1269->1267->1268->1284->1283->1282->1255->1253->1260->1259->1249->1246->963->964->962->1002->952->1000->998->994->995->996->987->988->979->980->969->977->989->990->991->2465->2466->2384->2382->2385->2379->2380->2445->2444->2405->2406->2398->2395->2397->2142->2141->2125->2126->2082->2080->2071->1979->1975->1967->1966->1974->1973->1971->1970->1948->1937->1939->1935->1931->1934->1673->1675->1674->1837->1671->1828->1825->1817->1815->1634->1814->1813->1632->1631->1742->1741->1740->1739->1591->1689->1585->1584->1688->1579->1679->1677->104->5680->5418->5431->5425->5424->5422->5413->5412->5411->66->5392->5391->5388->5291->5278->5289->5290->5283->5284->5280->50
Shortest distance: 150335.55441905273
Total energy cost: 259087.0


#**Question 2**

##**Task 1**

In [ ]:
from enum import Enum, auto
from typing import *

class Gridworld:
    State = Tuple[int, int]

    class Action(Enum):
        Up = auto()
        Down = auto()
        Left = auto()
        Right = auto()

    def __init__(self, noise: float, living_reward: float, grid: Tuple[Tuple[Any, ...], ...]):
        self.__noise = noise
        self.__living_reward = living_reward
        self.__n = len(grid)
        self.__m = len(grid[0])
        self.__grid = grid
        self.__states = {(x, y) for x in range(self.__n) for y in range(self.__m) if
                         grid[x][y] in (' ', 'S')}

    @property
    def states(self) -> Set[State]:
        return self.__states

    def get_actions(self, state: State) -> Set[Action]:
        x, y = state
        if x < 0 or x >= self.__n or y < 0 or y >= self.__m:
            raise ValueError('Not a valid state')
        if isinstance(self.__grid[state[0]][state[1]], float):  # Return no actions if terminal state
            return set()
        return {*Gridworld.Action}

    def _do_action(self, state: State, action: Action) -> State: # Returns the new state
        x, y = state

        if action == Gridworld.Action.Up:
            target_x, target_y = x - 1, y
        elif action == Gridworld.Action.Down:
            target_x, target_y = x + 1, y
        elif action == Gridworld.Action.Left:
            target_x, target_y = x, y - 1
        else:
            target_x, target_y = x, y + 1

        if target_x < 0 or target_x >= self.__n or target_y < 0 or target_y >= self.__m or \
                self.__grid[target_x][target_y] == '#':
            return state
        return target_x, target_y

    def get_transitions(self, current_state: State, action: Action) -> Dict[State, float]: # Returns a dictionary of "next_state": probability
        if action not in self.get_actions(current_state):
            raise ValueError('not a valid action')

        if self.__noise <= 0.:
            return {self._do_action(current_state, action): 1.}

        remaining = self.__noise / 2.
        if action in (Gridworld.Action.Up, Gridworld.Action.Down):
            outcomes = (
                (self._do_action(current_state, action), 1 - self.__noise),
                (self._do_action(current_state, Gridworld.Action.Left), remaining),
                (self._do_action(current_state, Gridworld.Action.Right), remaining)
            )
        else:
            outcomes = (
                (self._do_action(current_state, action), 1 - self.__noise),
                (self._do_action(current_state, Gridworld.Action.Up), remaining),
                (self._do_action(current_state, Gridworld.Action.Down), remaining)
            )

        transitions = {}
        for outcome, val in outcomes:
            transitions[outcome] = transitions.get(outcome, 0.) + val
        return transitions

    def get_reward(self, current_state: State, action: Action, next_state: State) -> float:
        if next_state not in self.get_transitions(current_state, action):
            raise ValueError('next state is not reachable from current state')
        grid_value = self.__grid[next_state[0]][next_state[1]]
        return grid_value if isinstance(grid_value, float) else self.__living_reward

In [ ]:
class ValueIterationAgent:
    def __init__(self, gridworld: Gridworld, discount_factor: float, iterations: int = 100):
        self.gridworld = gridworld
        self.discount_factor = discount_factor
        self.iterations = iterations
        self.values = {state: 0.0 for state in gridworld.states}

    def calc_q_value(self, state: Gridworld.State, action: Gridworld.Action) -> float:
        q_value = 0.0
        transitions = self.gridworld.get_transitions(state, action)
        for next_state, probability in transitions.items():
            q_value += probability * (
                self.gridworld.get_reward(state, action, next_state)
                + self.discount_factor * self.values.get(next_state, 0.0)
            )
        return q_value

    def iterate(self):
        for _ in range(self.iterations):
            new_values = {}
            for state in self.gridworld.states:
                actions = self.gridworld.get_actions(state)
                if actions:
                    new_values[state] = max(
                        self.calc_q_value(state, action) for action in actions
                    )
                else:
                    new_values[state] = 0.0
            self.values = new_values

    def getPolicy(self, state: Gridworld.State):
        actions = self.gridworld.get_actions(state)
        if not actions:
            return None
        return max(actions, key=lambda a: self.calc_q_value(state, a))

    def getValue(self, state: Gridworld.State) -> float:
        return self.values[state]

class PolicyIterationAgent(ValueIterationAgent):
    def __init__(self, gridworld: Gridworld, discount_factor: float, iterations: int = 100):
        super().__init__(gridworld, discount_factor, iterations)
        # Initial policy: always go Right
        self.policy = {}
        for state in gridworld.states:
            actions = gridworld.get_actions(state)
            self.policy[state] = Gridworld.Action.Right if actions else None

    def _policy_evaluation(self):
        for _ in range(self.iterations):
            new_values = {}
            for state in self.gridworld.states:
                action = self.policy[state]
                if action is None:
                    new_values[state] = self.values[state]
                else:
                    new_values[state] = self.calc_q_value(state, action)
            self.values = new_values

    def _policy_improvement(self) -> bool:
        policy_changed = False
        for state in self.gridworld.states:
            actions = self.gridworld.get_actions(state)
            if not actions:
                continue
            best_action = max(actions, key=lambda a: self.calc_q_value(state, a))
            if best_action != self.policy[state]:
                self.policy[state] = best_action
                policy_changed = True
        return policy_changed

    def iterate(self):
        while True:
            self._policy_evaluation()
            changed = self._policy_improvement()
            if not changed:
                break

    def getPolicy(self, state: Gridworld.State):
        actions = self.gridworld.get_actions(state)
        if not actions:
            return None
        return self.policy[state]

    def getValue(self, state: Gridworld.State) -> float:
        return self.values[state]



def print_value_function(agent: ValueIterationAgent, grid: tuple, label: str):
    n, m = len(grid), len(grid[0])
    print(f"\n{'='*55}")
    print(f"  Value Function — {label}")
    print(f"{'='*55}")
    for x in range(n):
        row = ""
        for y in range(m):
            cell = grid[x][y]
            if cell == '#':
                row += "  ##### "
            elif isinstance(cell, float):
                row += f" [{cell:+5.1f}]"
            else:
                row += f"  {agent.getValue((x, y)):+5.3f}"
        print(row)


def print_policy(agent: ValueIterationAgent, grid: tuple, label: str):
    symbols = {
        Gridworld.Action.Up:    "↑",
        Gridworld.Action.Down:  "↓",
        Gridworld.Action.Left:  "←",
        Gridworld.Action.Right: "→",
        None:                   "X",
    }
    n, m = len(grid), len(grid[0])
    print(f"\n{'='*55}")
    print(f"  Policy — {label}")
    print(f"{'='*55}")
    for x in range(n):
        row = ""
        for y in range(m):
            cell = grid[x][y]
            if cell == '#':
                row += "  # "
            elif isinstance(cell, float):
                row += "  X "
            else:
                a = agent.getPolicy((x, y))
                row += f"  {symbols[a]} "
        print(row)



def main():
    grid = (
        (' ', ' ', ' ', ' ', 10.),
        (' ', ' ', ' ', ' ', ' '),
        (' ', '#', ' ', '#', ' ',),
        (' ', ' ', ' ', ' ', ' '),
        ('S', ' ', ' ', ' ', ' '),
    )
    game = Gridworld(noise=0.2, living_reward=-0.1, grid=grid)

    vi = ValueIterationAgent(game, discount_factor=0.9, iterations=200)
    vi.iterate()
    print_value_function(vi, grid, "Value Iteration")
    print_policy(vi, grid, "Value Iteration")

    pi = PolicyIterationAgent(game, discount_factor=0.9, iterations=200)
    pi.iterate()
    print_value_function(pi, grid, "Policy Iteration")
    print_policy(pi, grid, "Policy Iteration")

if __name__ == '__main__':
    main()


  Value Function — Value Iteration
  +5.974  +7.012  +8.209  +9.604 [+10.0]
  +5.422  +6.343  +7.279  +8.439  +9.604
  +4.639  #####   +6.269  #####   +8.311
  +4.008  +4.523  +5.360  +5.991  +7.058
  +3.453  +3.966  +4.586  +5.223  +5.991

  Policy — Value Iteration
  →   →   →   →   X 
  →   →   →   →   ↑ 
  ↑   #   ↑   #   ↑ 
  ↑   →   ↑   →   ↑ 
  ↑   →   ↑   →   ↑ 

  Value Function — Policy Iteration
  +5.974  +7.012  +8.209  +9.604 [+10.0]
  +5.422  +6.343  +7.279  +8.439  +9.604
  +4.639  #####   +6.269  #####   +8.311
  +4.008  +4.523  +5.360  +5.991  +7.058
  +3.453  +3.966  +4.586  +5.223  +5.991

  Policy — Policy Iteration
  →   →   →   →   X 
  →   →   →   →   ↑ 
  ↑   #   ↑   #   ↑ 
  ↑   →   ↑   →   ↑ 
  ↑   →   ↑   →   ↑ 


##**Task 2: Monte Carlo Prediction and Control**

Define grid-world settings

In [ ]:
import random
from collections import defaultdict

# Grid settings
GRID_SIZE = 5
START_STATE = (0, 0)
GOAL_STATE = (4, 4)

# Use the roadblocks shown in the figure
OBSTACLES = {(1, 2), (3, 2)}

# If your instructor wants the text version instead, use:
# OBSTACLES = {(2, 1), (2, 3)}

ACTIONS = ["U", "D", "L", "R"]

# Hyperparameters
GAMMA = 0.9
EPSILON = 0.1
NUM_EPISODES = 500000
MAX_STEPS_PER_EPISODE = 200

# Task 2 default: use stochastic dynamics from the prompt
STOCHASTIC = False

# Movement mapping
ACTION_DELTA = {
    "U": (0, 1),
    "D": (0, -1),
    "L": (-1, 0),
    "R": (1, 0)
}

Build the state space

In [ ]:
states = []
for x in range(GRID_SIZE):
    for y in range(GRID_SIZE):
        if (x, y) not in OBSTACLES:
            states.append((x, y))

non_terminal_states = [s for s in states if s != GOAL_STATE]

Helper function to move in the grid

In [ ]:
def move(state, action):
    if state == GOAL_STATE:
        return state

    x, y = state
    dx, dy = ACTION_DELTA[action]
    nx, ny = x + dx, y + dy

    # Out of bounds
    if nx < 0 or nx >= GRID_SIZE or ny < 0 or ny >= GRID_SIZE:
        return state

    # Into obstacle
    if (nx, ny) in OBSTACLES:
        return state

    return (nx, ny)

Environment step function

In [ ]:
def step(state, action):
    if state == GOAL_STATE:
        return state, 0, True

    if STOCHASTIC:
        perpendicular = {
            "U": ["L", "R"],
            "D": ["L", "R"],
            "L": ["U", "D"],
            "R": ["U", "D"]
        }

        r = random.random()
        if r < 0.8:
            actual_action = action
        elif r < 0.9:
            actual_action = perpendicular[action][0]
        else:
            actual_action = perpendicular[action][1]
    else:
        actual_action = action

    next_state = move(state, actual_action)

    if next_state == GOAL_STATE:
        return next_state, 10, True
    else:
        return next_state, -1, False

ε-greedy action selection

In [ ]:
def epsilon_greedy_action(Q, state, epsilon=EPSILON):
    if random.random() < epsilon:
        return random.choice(ACTIONS)

    q_values = Q[state]
    max_q = max(q_values[a] for a in ACTIONS)
    best_actions = [a for a in ACTIONS if q_values[a] == max_q]
    return random.choice(best_actions)

Generate one full episode

In [ ]:
def generate_episode(Q):
    episode = []
    state = START_STATE

    for _ in range(MAX_STEPS_PER_EPISODE):
        action = epsilon_greedy_action(Q, state, EPSILON)
        next_state, reward, done = step(state, action)

        episode.append((state, action, reward))
        state = next_state

        if done:
            break

    return episode

First-visit Monte Carlo control

In [ ]:
def monte_carlo_control():
    # Q(s, a)
    Q = defaultdict(lambda: {a: 0.0 for a in ACTIONS})

    # For averaging returns
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for episode_idx in range(NUM_EPISODES):
        episode = generate_episode(Q)

        G = 0
        visited_sa = set()

        # Traverse backwards to compute returns
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = GAMMA * G + reward

            # First-visit MC
            if (state, action) not in visited_sa:
                visited_sa.add((state, action))

                returns_sum[(state, action)] += G
                returns_count[(state, action)] += 1
                Q[state][action] = returns_sum[(state, action)] / returns_count[(state, action)]

    # Extract greedy policy from Q
    policy = {}
    for state in non_terminal_states:
        max_q = max(Q[state][a] for a in ACTIONS)
        best_actions = [a for a in ACTIONS if Q[state][a] == max_q]
        policy[state] = random.choice(best_actions)

    return Q, policy

Compute state values from Q

In [ ]:
def compute_state_values(Q):
    V = {}

    for state in non_terminal_states:
        V[state] = max(Q[state][a] for a in ACTIONS)

    V[GOAL_STATE] = 0.0
    return V

Print the learned policy as a grid

In [ ]:
def print_policy(policy):
    arrow = {
        "U": "↑",
        "D": "↓",
        "L": "←",
        "R": "→"
    }

    print("=" * 58)
    print("    Policy - Monte Carlo")
    print("=" * 58)
    print()

    for y in range(GRID_SIZE - 1, -1, -1):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)

            if s in OBSTACLES:
                row.append("#")
            elif s == GOAL_STATE:
                row.append("X")
            else:
                row.append(arrow[policy[s]])

        print("  ".join(f"{cell:^3}" for cell in row))
    print()

Print the learned state values

In [ ]:
def print_values(V):
    print("=" * 58)
    print("    Value Function - Monte Carlo")
    print("=" * 58)

    for y in range(GRID_SIZE - 1, -1, -1):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)

            if s in OBSTACLES:
                row.append(" ##### ")
            elif s == GOAL_STATE:
                row.append("[+10.0]")
            else:
                row.append(f"{V[s]:+6.3f}")

        print("  ".join(row))
    print()

Optional comparison with Task 1 policy

In [ ]:
def compare_policies(mc_policy, optimal_policy_task1):
    same = 0
    total = 0

    for state in non_terminal_states:
        if state in optimal_policy_task1:
            total += 1
            if mc_policy[state] == optimal_policy_task1[state]:
                same += 1

    print("\nComparison with Task 1 optimal policy:")
    print(f"Matching states: {same}/{total}")
    print(f"Match percentage: {(same / total) * 100:.2f}%")

Run Task 2

In [ ]:
random.seed(42)

Q_mc, policy_mc = monte_carlo_control()
V_mc = compute_state_values(Q_mc)

# Make goal display match the screenshot style
V_mc[GOAL_STATE] = 10.0

print("Training Monte Carlo for 500000 episodes...\n")
print("Learned Policy (Task 2):\n")

print_values(V_mc)
print_policy(policy_mc)

Training Monte Carlo for 500000 episodes...

Learned Policy (Task 2):

    Value Function - Monte Carlo
+4.093  +5.870  +7.857  +10.000  [+10.0]
+2.601  +4.112  +5.945  +7.859  +10.000
+1.231   #####   +4.196   #####   +7.861
-0.008  +1.234  +2.642  +4.066  +5.908
-1.066  +0.009  +1.235  +2.541  +4.075

    Policy - Monte Carlo

 →    →    →    →    X 
 →    ↑    →    ↑    ↑ 
 ↑    #    ↑    #    ↑ 
 ↑    →    ↑    →    ↑ 
 →    →    ↑    →    ↑ 



##**Task 3**

In [ ]:
import numpy as np
import random

In [ ]:
# --- Grid World Environment Configuration ---
WIDTH, HEIGHT = 5, 5
START = (0, 0)
GOAL = (4, 4)
BLOCKS = [(2, 1), (2, 3)]
ACTIONS = ['Up', 'Down', 'Left', 'Right']

In [ ]:
# --- Hyperparameters as per Lab Manual ---
ALPHA = 0.1   # Learning rate
EPSILON = 0.1 # Exploration rate
GAMMA = 0.9   # Discount factor
EPISODES = 10000

In [ ]:
ARROW_MAP = {
    'Up': '↑',    # Points to higher y-coordinates
    'Down': '↓',  # Points to lower y-coordinates
    'Left': '←',  # Points to lower x-coordinates
    'Right': '→'  # Points to higher x-coordinates
}

In [ ]:
class QLearningAgent:
    def __init__(self):
        # Tabular representation: 5x5 grid with 4 possible actions per state
        self.q_table = np.zeros((WIDTH, HEIGHT, len(ACTIONS)))
        self.action_map = {a: i for i, a in enumerate(ACTIONS)}

    def get_next_state(self, state, action):
        """Implements the Stochastic Transition Model."""
        x, y = state
        r = random.random()

        # 0.8 probability for intended, 0.1 each for perpendicular
        if r < 0.8:
            actual_move = action
        elif r < 0.9:
            actual_move = self._perp_left(action)
        else:
            actual_move = self._perp_right(action)

        nx, ny = x, y
        if actual_move == 'Up': ny += 1
        elif actual_move == 'Down': ny -= 1
        elif actual_move == 'Left': nx -= 1
        elif actual_move == 'Right': nx += 1

        # If move is out of bounds or hits a roadblock, stay in current state
        if 0 <= nx < WIDTH and 0 <= ny < HEIGHT and (nx, ny) not in BLOCKS:
            return (nx, ny)
        return state

    def _perp_left(self, a):
        return {'Up': 'Left', 'Down': 'Right', 'Left': 'Down', 'Right': 'Up'}[a]

    def _perp_right(self, a):
        return {'Up': 'Right', 'Down': 'Left', 'Left': 'Up', 'Right': 'Down'}[a]

    def choose_action(self, state):
        """Epsilon-greedy strategy."""
        if random.random() < EPSILON:
            return random.choice(ACTIONS)

        # Exploit: pick the best Q-value (with random tie-breaking)
        state_qs = self.q_table[state[0], state[1], :]
        max_indices = np.where(state_qs == np.max(state_qs))[0]
        return ACTIONS[random.choice(max_indices)]

    def train(self):
        print(f"Training Q-Learning for {EPISODES} episodes...")
        for _ in range(EPISODES):
            state = START
            while state != GOAL:
                action = self.choose_action(state)
                next_state = self.get_next_state(state, action)

                # Rewards: -1 per step, +10 for Goal
                reward = 10 if next_state == GOAL else -1

                # Q-Learning Update Rule
                a_idx = self.action_map[action]
                best_next_q = np.max(self.q_table[next_state[0], next_state[1], :])

                self.q_table[state[0], state[1], a_idx] += ALPHA * (
                    reward + GAMMA * best_next_q - self.q_table[state[0], state[1], a_idx]
                )
                state = next_state

    def display_results(self):
        # 1. Value Function Output
        print("=======================================================")
        print("  Value Function — Q-Learning")
        print("=======================================================")
        # To make (2,1) and (2,3) side-by-side, we iterate through X as rows
        # We go from x=4 down to 0 to keep the goal area at the top visually
        for x in range(WIDTH - 1, -1, -1):
            row = []
            for y in range(HEIGHT): # y increases from left to right (columns)
                state = (x, y)
                if state == GOAL:
                    row.append("[+10.0]")
                elif state in BLOCKS:
                    row.append(" ##### ")
                else:
                    v = np.max(self.q_table[x, y, :])
                    row.append(f"{v: >+7.3f}")
            print("  " + "  ".join(row))

        # 2. Policy Output
        print("\n=======================================================")
        print("  Policy — Q-Learning")
        print("=======================================================")
        for x in range(WIDTH - 1, -1, -1):
            row = []
            for y in range(HEIGHT):
                state = (x, y)
                if state == GOAL:
                    row.append("X")
                elif state in BLOCKS:
                    row.append("#")
                else:
                    best_a_idx = np.argmax(self.q_table[x, y, :])
                    best_a = ACTIONS[best_a_idx]
                    row.append(ARROW_MAP[best_a])
            print("  " + "   ".join(row) + " ")


In [ ]:
if __name__ == "__main__":
    agent = QLearningAgent()
    agent.train()
    print("\nLearned Policy (Task 3):")
    agent.display_results()

Training Q-Learning for 10000 episodes...

Learned Policy (Task 3):
  Value Function — Q-Learning
   +2.663   +4.368   +6.522   +9.309  [+10.0]
   +1.350   +2.716   +4.458   +6.563   +9.217
   +0.080   #####    +3.111   #####    +6.419
   -1.011   +0.092   +1.558   +2.797   +4.199
   -1.859   -0.852   +0.554   +1.528   +2.601

  Policy — Q-Learning
  ↑   ↑   ↑   ↑   X 
  →   →   →   ↑   → 
  →   #   →   #   → 
  →   ↑   ↑   ↑   → 
  →   ↑   ↑   ↑   → 
